# Maia-2 Model Test
A quick test of the Maia-2 model to see how well it predicts my val games

In [3]:
# Dowload the Model
from maia2 import model, inference

maia = model.from_pretrained(type="rapid", device="auto", save_root="/Users/logancowley/Documents/BYUI/Fall_2026/CSE455/chess/models/maia2")
prepared = inference.prepare() 

Downloading...
From (original): https://drive.google.com/uc?id=1gbC1-c7c0EQOPPAVpGWubezeEW8grVwc
From (redirected): https://drive.google.com/uc?id=1gbC1-c7c0EQOPPAVpGWubezeEW8grVwc&confirm=t&uuid=8494400a-96b3-4053-b93b-beb5d9203c5b
To: /Users/logancowley/Documents/BYUI/Fall_2026/CSE455/chess/models/maia2/.rapid_model.pt._lbgmdtr.part
100%|██████████| 280M/280M [00:08<00:00, 34.5MB/s] 


Model for rapid games loaded to mps.


In [11]:
# Predict one position
fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"  # after 1.e4

move_probs, win_prob = inference.inference_each(
    maia, prepared, fen,
    elo_self=1000,   # rating of the player whose turn it is
    elo_oppo=1000,   # rating of their opponent
)

print(list(move_probs.items())[:5])   # top 5 moves, most likely first
print(win_prob)

[('g8f6', 0.3756), ('b8c6', 0.2084), ('g8h6', 0.075), ('d7d5', 0.0547), ('e7e5', 0.0478)]
0.6715


In [9]:
# Go Through a Game Move by Move
import io, chess.pgn

pgn_text = """[White "whopizza1"]
[Black "opponent"]
[WhiteElo "1000"]
[BlackElo "1050"]

1. e4 e5 2. Nf3 Nc6 3. Bc4 Bc5 4. c3 Nf6 5. d4 exd4 *"""

game = chess.pgn.read_game(io.StringIO(pgn_text))
w_elo, b_elo = int(game.headers["WhiteElo"]), int(game.headers["BlackElo"])
board = game.board()

hits = total = 0
for move in game.mainline_moves():
    me, opp = (w_elo, b_elo) if board.turn == chess.WHITE else (b_elo, w_elo)
    probs, _ = inference.inference_each(maia, prepared, board.fen(), me, opp)
    guess = next(iter(probs))
    hits += guess == move.uci()
    total += 1
    print(f"{board.fullmove_number:>3} played {move.uci():6} maia {guess:6} p={probs[guess]:.2f}")
    board.push(move)

print(f"top-1 accuracy: {hits}/{total}")

  1 played e2e4   maia g1f3   p=0.33
  1 played e7e5   maia g8f6   p=0.36
  2 played g1f3   maia e1e2   p=0.22
  2 played b8c6   maia b8c6   p=0.15
  3 played f1c4   maia b1c3   p=0.22
  3 played f8c5   maia g8f6   p=0.34
  4 played c2c3   maia d2d3   p=0.23
  4 played g8f6   maia g8f6   p=0.54
  5 played d2d4   maia d2d4   p=0.43
  5 played e5d4   maia e5d4   p=0.70
top-1 accuracy: 4/10


In [10]:
import pandas as pd

df = pd.DataFrame({
    "board":        [fen],
    "move":         ["e7e5"],
    "active_elo":   [1000],
    "opponent_elo": [1000],
})

df, acc = inference.inference_batch(df, maia, verbose=1, batch_size=512, num_workers=0)
print(acc)        # top-1 accuracy over the rows
df.move_probs     # a full probability dictionary for each row

100%|██████████| 1/1 [00:00<00:00,  6.14it/s]

0.0


0    {'g8f6': 0.3611, 'b8c6': 0.2152, 'g8h6': 0.078...
Name: move_probs, dtype: object